# Introduction
For this assignement we will use the data from application 2 of this [this](https://www.sciencedirect.com/science/article/abs/pii/S1053811914002249) paper. For the purpose of speed we will only use the first 5 participants of the data (see the [HMP](https://direct.mit.edu/imag/article/doi/10.1162/imag_a_00400/125469) paper for the method and [https://github.com/GWeindel/man_hmp/tree/main/results/replication/SAT_data](https://github.com/GWeindel/man_hmp/tree/main/results/replication/SAT_data) for the whole (preprocessed) data).

In this experiment, participants performed a random-dot motion task. They were asked to indicate the direction of motion of a cloud of moving dots. While a proportion of the dots moved in a target direction, the remainder moved randomly and makes the direction discrimination more difficult. Difficulty of the task was calibrated per subject. Prior to each trial, participants received a cue that indicated whether they should respond as quickly as possible or whether they should focus on giving an accurate response: the 'speed' and 'accuracy' conditions. In this tutorial we will ignore the difference between these conditions, but in the next tutorial we will look at how we can take conditions into account in the HMP analysis.

## Data preparation

First, we load the required packages and download the data. Depending on your internet connection this can take some time

In [ ]:
import os
import requests
import numpy as np
import xarray as xr
import hmp
from mne.io import read_info
import matplotlib.pyplot as plt

# Declaring path where the EEG data will be stored
epoch_data_path = os.path.join('sample_data', 'eeg_example')
os.makedirs(epoch_data_path, exist_ok=True)

# URLs of the first 5 participants in the SAT experiment, navigate the osf folder and adapt those if you want to do this tutorial on other data (e.g. P3, N2pc)
file_urls = [
    "https://osf.io/download/67cffa85f67af67e7a92f0a6/",
    "https://osf.io/download/67cffa85f67af67e7a92f0a8/",
    "https://osf.io/download/67cffa85f67af67e7a92f0aa/",
    "https://osf.io/download/67cffa85f67af67e7a92f0ac/",
    "https://osf.io/download/67cffa85f67af67e7a92f0ae/",
]

# Download and save each file if not already in folder
for i, url in enumerate(file_urls, start=1):
    file_path = os.path.join(epoch_data_path, f'participant{i}_epo.fif')
    if not os.path.exists(file_path):
        response = requests.get(url)
        with open(file_path, 'wb') as f:
            f.write(response.content)

# Recovering individual files and participant names
subj_files = [os.path.join(epoch_data_path, f) for f in os.listdir(epoch_data_path) if f.endswith('.fif')]  # Create a list of files with full paths
subj_names = [os.path.splitext(f)[0] for f in os.listdir(epoch_data_path) if f.endswith('.fif')]  # Extract subject names based on file names

# Recovering channel information (assuming the same for all participant)
info = read_info(subj_files[0], verbose=False)

# Classical EEG analyses

This first part is just an introduction to classical event related potential analyses for EEG. For this part we need the mne python package that should be present if you installed HMP.

In [ ]:
import mne

For simplicity we will only analyze one participant but what you'll be seeing in this part is easily adaptable to multiple participants

In [ ]:
# Function to read EEG data that has been epoched around stimulus:
epochs = mne.read_epochs(subj_files[0])
epochs._data /= 1e6 #Just converting to V instead of microV
epochs = epochs.crop(tmin=0, tmax=1) #Cropping epochs to 1 second for illustration

In order to better understand the data we have at hand we can look up the `epochs.info`

In [ ]:
epochs.info

We learn that the data is made of 33 EEG channels, sampled at 500Hz, with a bandpass filter between 1 and 35Hz. The EEG electrodes are (in theory) placed on the scalp as illustrated below:

In [ ]:
epochs.plot_sensors(show_names=True);


The metadata attribute stores behavior relevant information for each epoch

In [ ]:
epochs.metadata

Where `resp` indicates the response side, `RT` the reaction time, `cue` the condition (be fast `SP` or accurate `AC`) and `movement` the direction of the coherently moving dots in the cloud of dots on the screen left or to the right.

## Looking at the time domain

The data of this participant is made of 186 trials, across the 30 channels illustrated above and on 501 samples (1 sec sampled at 500Hz

In [ ]:
epochs._data.shape

If we inspect a single trial the data is too noisy to be informative by visalization alon: 

In [ ]:
plt.plot(epochs._data[0,:,:].T)# Selecting first trial
plt.ylabel('voltage')
plt.xlabel('Time (in sample)');

A seemingly good idea is to average these trials, hoping to cancel out the noise not related to stimulus presentation:

In [ ]:
epochs.plot_image(picks=["Cz"]);

Here the average of the electrode Cz shows some consistent variations in trials (blue band in the top plot) reflected in the average ERP.

### Exercise
> Knowing the content of the lecture, what is the problem of this representation?

[your response here]

## Looking at the spatial domain

Instead of looking at a single electrode we can also combine time and space by looking at slices of time and averaging the values of all electrode for a particular time point 

In [ ]:
epochs.average().plot_joint(times=[0.99]);

The top right plots shows you the average of all electrodes (topomap) at time 0.99 seconds

### Exercise

> Using the times argument of the function above, show the topography for a better timepoint than the one illustrated above

Now traditional analyses are very limited as we have make strong assumptions on the exact (single-trial) timing of the events. Furthermore, it is difficult to illustrate both time and space, even harder is to perfrorm inferences on these dimensions (but see other [MNE tutorial](https://mne.tools/stable/auto_tutorials/index.html) for those who are interested) 

# HMP analyses

From now on we will use the HMP method precented in the lecture

## Reading the data

We use the following HMP function to read the data for all participants

In [ ]:
# At what frequency we want the data, upsample only if you have a large amount of RAM (and time)
sfreq = 250

# Then we read the data as shown in Tutorial 1
epoch_data = hmp.io.read_mne_data( subj_files, data_format='epochs', sfreq=sfreq,
                            lower_limit_rt=0.2, upper_limit_rt=2,
                            rt_col = 'RT', scale = 1000, 
                            verbose=False, subj_name=subj_names )
epoch_data['data'] /= 1e6

This object now contains all the data of all 5 participants along with the metadata

In [ ]:
print(epoch_data)

Contrary to the plots above, this time we don't epoch on a fixed window but instead we only record the EEG from stimulus to response (here we use the [xarray](https://docs.xarray.dev/en/stable/user-guide/index.html) python package, a handy way of using matrices with named dimensions).

In [ ]:
epoch_data.sel(participant='participant3_epo', epoch=0).data.plot.line(x='sample')
plt.xlim(0,501)
plt.show()

Thus we remove information that is not related to the realization of the task, here a decision-making task.

## Preprocessing

At this point we have the epoched EEG data (based on RT) with 30 channels, which we need to transform to principal component (PC) space using a PCA as a spatial filter. When transforming the data, a prompt requests how many PCs we want to keep. The best is to maximise the number of components kept so that we don't lose information. However, the more PCs we include the more computationally intensive the estimations will be. For this two rules of thumb exists, taking PCs that explain x% of the variance of the data or select the PC number at which the explained variance seems to almost stop decreasing (i.e. the "elbow" method).

The HMP `preprocessing` module will show you the two graphs below, and asks you how many PCs you would like to retain:

In [ ]:
preprocessed = hmp.preprocessing.Standard(epoch_data)

In this case we see that 99% explained variance is achieved with 6 PCs, and that this is coherent with when the explained variance really drops (with a larger number of electrodes than the 30 used here, 99% might be achieved with more PCs).

In [ ]:
print(preprocessed.data)

The data is now arranged as 6 principal components x 64690 samples: all trials of all participants were concatenated for the remainder of the analysis.

### Exercise:
> The PCA operation gives you weights for each electrode, given the hints below, what operation do you need to apply in order to obtain a timeserie for the first PC?

In [ ]:
weights = preprocessed.weights.sel(component = 0) # The weights organized as loadings of each channels to the 1st PC
data = epoch_data.sel(participant='participant3_epo', epoch=0).data # The data of a single trial organized as channels X samples
plt.plot();#Fill in the correct mathematical operation

Now that the data has been spatially filtered using the PCA we will cross-correlate it with the expected pattern at the single-trial level:

In [ ]:
# Defining the expected HMP pattern, in this case a half sine of 10Hz, thus 50 ms duration
event_properties = hmp.patterns.HalfSine.create_expected(sfreq=epoch_data.sfreq, width=50)
# Performing the crosscorrelation between the preprocessed data and the expected pattern
trial_data = hmp.trialdata.TrialData.from_preprocessed(preprocessed=preprocessed, pattern=event_properties.template)

In [ ]:
plt.plot(preprocessed.data.sel(participant='participant3_epo', epoch=0).T);

The data is now cross-correlated with the pattern as introduced in the lecture (see also this section of [Tutorial 1](https://hmp.readthedocs.io/en/latest/notebooks/1-How_HMP_works.html#Data-preprocessing)) and ready to be fitted!

### Exercise:
> To which very classical machine learning implemented in convolutional neural networks can the crosscorrelation be associated? 

[your response here]

This time we represent the same trial as 6 lines (one for each PC) across all the samples. This is the data that we use to estimat HMP models

## Fitting

As introduced during the lecture (see also [Tutorial 2](https://hmp.readthedocs.io/en/latest/notebooks/2-The_different_model_classes.html)), the ```CumulativeEstimation``` method starts by sliding a candidate event from 0 to mean RT. When an event is found – the Expectation Maximization estimation converges – one event is added to the model and the slide continues. This way we can detect new events while accounting for the previous ones. 

In [ ]:
model = hmp.models.CumulativeMethod(event_properties)
_, estimates_cumulative = model.fit_transform(trial_data)

The method found 4 events, we can represent their average timecourse (in samples not in seconds) and topography with the function below

In [ ]:
hmp.visu.plot_topo_timecourse(epoch_data, estimates_cumulative, info, as_time=False)

### Exercise:

> The parameters we saw in the lecture are contained in the `time_pars` and `channel_pars` attribute of the model. Illustrate the multivariate part of the model, as seen in class (see also  [Tutorial 1](https://hmp.readthedocs.io/en/latest/notebooks/1-How_HMP_works.html#))

```
TIP:
Use the attribute `model.final_model.channel_pars.T`

In [ ]:
...

Thanks to these parameters we can turn the EEG activity of a single trial into probability space

In [ ]:
plt.plot(estimates_cumulative.unstack().sel(participant='participant3_epo', epoch=0))

### Exercise
> In the lecture we discussed how we identify the most likely times of each event for a particular trial. Show what these times are for this trial based on the code above that extract probabilities for a specific trial

    TIP: you can use the `np.argmax` method

In [ ]:
...;

But this is just for a particular model, we can try alternative ones if for some reason we expect more or less events.

## Exploring different solutions

An HMP model can also be initialized with a predefined number of events using the `hmp.models.EventModel` class.

### Exercise: 

> Change the number of estimated events by specifing it, see for example [here(https://hmp.readthedocs.io/en/latest/notebooks/2-The_different_model_classes.html#Model-with-a-fixed-number-of-event-with-EventModel) and represent their average timecourse and topographies as for the other model

In [ ]:
...;

## Analyzing condition effect on interval between events

Going back to our 4 event model (ensure that `estimates_cumulative` refers to the initial model given in the assignement). As seen in the lecture we can use the by-trial derived times as above to study the differences between experimental conditions, stimuli or participants. To compute the trial we use the following function:

In [ ]:
# Compute max likely time for each trial and each event
times =  hmp.utils.event_times(estimates_cumulative, duration=True, add_rt=True, as_time=True)
times

This is literraly just doing `argmax` for every trial and every event. This format is not necessarily an easy format to work with so we convert it to a probably more familiar format: `pd.DataFrame`

In [ ]:
# Turn into dataframe and recover metadata
times = times.unstack().to_dataframe(name='duration')
times = times[~times.duration.isna()]  #Remove rejected trials
times = times.reset_index().set_index(['participant','epoch'])

# Recover metadata and merge with times
times_metadata = epoch_data.sel(sample=0, channel='Cz').to_dataframe().iloc[:,3:]
times_metadata = times_metadata.reset_index().set_index(['participant','epoch'])
times = times.merge(times_metadata, on=['participant','epoch']).reset_index()

times

And we can now process this data as you would for a typical pandas dataframe:

In [ ]:
mean_ac = times[times.cue == 'AC'].groupby(['event']).duration.mean()
mean_sp = times[times.cue == 'SP'].groupby(['event']).duration.mean()

plt.plot(mean_ac.index, mean_ac.values, 'o-', label='AC')
plt.plot(mean_sp.index, mean_sp.values, 'o-', label='SP')
plt.legend()
plt.xlabel('Event')
plt.ylabel('Duration (ms)')
plt.title('Event durations by condition')

### Exercise:
> Change the code above to display the difference in the time between events between two randomly selected participants

In [ ]:
...;

## Comparing conditions on centered ERPs:

In the lecture it was shown that we can use the information of the single-trial to plot event related potentials around the time of an event of interest. Here is an example where we center each trial around `event` 2 and display the time-courses of the average of electrodes `CP1` and `CP2` between the two speed/accuracy instructions:

In [ ]:
# Plotting centered ERPs for each condition (AC and SP) with confidence intervals (±1 std)
fig, ax = plt.subplots(1,1)

# Get event times (positions not durations) for all events/trials, including stimulus onset
times_position = hmp.utils.event_times(estimates_cumulative, duration=False, mean=False, add_stim=True, as_time=False)

# Define window size in samples
baseline = -.1*sfreq  # 100 ms before event
n_samples = .4*sfreq  # 400 ms window

event = 2  # Event index to center on, 0 is stimulus

# Select a subset of channels to analyze (e.g., centroparietal channels)
channel_subset = ['CP1', 'CP2']

for SAT in ["AC","SP"]:
    # Select trials for the current condition and stack participant/epoch as 'trial' for easiness
    subset = epoch_data.where((epoch_data.cue == SAT), drop=True).stack({'trial':['participant','epoch']}).data.dropna('trial', how="all")
    # Center activity on the event for selected channels
    centered = hmp.utils.centered_activity(subset, times_position, channel_subset,
        event=event, n_samples=n_samples, baseline=baseline)
    # Average across channels,
    centered = centered.data.unstack().mean('channel')
    # Compute mean and std across participants
    indiv_traces = centered.groupby('participant').mean(dim='epoch')
    mean_tc = indiv_traces.mean('participant')
    std_tc = indiv_traces.std('participant')
    # Plot the timecourse with confidence interval
    ax.plot(centered.sample, mean_tc, label=SAT,)
    ax.fill_between(centered.sample, mean_tc-std_tc, mean_tc+std_tc, alpha=0.2)
ax.set_xlabel(f'Time from Event {event} (ms)')
ax.set_ylabel('Amplitude (V)')
plt.legend(title='Condition')
plt.tight_layout()

### Exercise:

Change the ERP plot to study the  difference between left vs. right responses. We expect that the electrodes, over the motor cortices (typically C3 and C4), controlateral to the responding hand should be negative (i.e. lateralized readiness potential) in a couple of hunderd of milliseconds BEFORE the event. By completing the code below, look what event displays this pattern using for example electrodes C3 OR C4:


### Bonus:
> Instead of picking either C3 or C4, what should you do to easily represent that the controlateral hemisphere is more negative?

In [ ]:
...;

# END

And that's it. If you want to look further on how to process this data, look at the [tutorial 3](https://hmp.readthedocs.io/en/latest/notebooks/3-Applying_HMP_to_real_data.html) of HMP that uses the same data